# Session B2: bigger training set + two-stage detection (GPU)

1. Retrains the YOLOv8s detector on the bigger dataset from Session A2 (119 train images), with a fixed
   35-epoch schedule (the first run's time-capped mode stretched the learning-rate schedule).
2. **Hard negative mining**: the detector's false positives on training tiles become extra negatives.
3. Trains an **EfficientNet-B0 classifier** (mitosis vs look-alike) on 128 px crops.
4. Evaluates detector only and detector + classifier on the **same 14 test images** as Session B.
   The scoring variant and threshold are chosen on val only.

**Before running**: Add Input -> Your Work -> **Session A2** notebook. Accelerator **GPU P100** (or T4),
Internet **On**. Then Save Version -> **Save & Run All (Commit)**. Expected run time about 1 hour.

In [ ]:
# Settings
EPOCHS = 35        # fixed schedule
HOURS  = 0.9       # safety cap: stops training, does not change the schedule
PATIENCE = 20
MODEL  = 'yolov8s.pt'
BATCH  = 16
CLS_EPOCHS = 8
WORK   = '/kaggle/working'
DATA   = '/tmp/midog_yolo'
CODE   = f'{WORK}/code'

# Session B (first run) results on the same test images, for the comparison table
BASELINE = {'name': 'Session B: 49 train images, detector only', 'precision': 0.697, 'recall': 0.669, 'f1': 0.682,
            'count_correlation': 0.9754}

In [ ]:
# Unpack Session A2's output
import glob, os, zipfile, shutil, json
from pathlib import Path
if not Path(f'{DATA}/data.yaml').exists():
    zips = glob.glob('/kaggle/input/**/midog_yolo_v2.zip', recursive=True)
    if not zips:
        raise FileNotFoundError('midog_yolo_v2.zip not found. Use "Add Input" and add the Session A2 notebook.')
    print('Unzipping', zips[0])
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall('/tmp')
yaml = Path(f'{DATA}/data.yaml').read_text().splitlines()
Path(f'{DATA}/data.yaml').write_text('\n'.join(f'path: {DATA}' if l.startswith('path:') else l for l in yaml) + '\n')
for s in ('train', 'val', 'test'):
    print(s, len(os.listdir(f'{DATA}/images/{s}')), 'tiles,', len(os.listdir(f'{DATA}/full/{s}')), 'full images')
for s in ('train', 'val'):
    print(s, 'crops:', {l: len(os.listdir(f'{DATA}/crops/{s}/{l}')) for l in ('mitosis', 'imposter')})

In [ ]:
!pip install -q ultralytics tifffile imagecodecs
!nvidia-smi --query-gpu=name,memory.total --format=csv
os.makedirs(CODE, exist_ok=True)

In [ ]:
%%writefile /kaggle/working/code/tiled_inference.py
"""
Tiled (sliding-window) inference on large pathology images.

A YOLO model trained on 640 x 640 tiles cannot see a ~6,600 x 4,900 px image
at once without shrinking cells to a few pixels. Instead we:
  1. cut the image into overlapping tiles,
  2. skip background tiles (mostly white glass),
  3. run the detector on tiles in batches,
  4. shift tile coordinates back to the full image,
  5. merge duplicate detections from overlapping tiles (center-distance NMS),
  6. report the mitotic count, a density heatmap and an annotated overview.

Usage (single image or folder)
  python tiled_inference.py --weights runs/midog/midog_yolov8s/weights/best.pt \
      --source datasets/midog_yolo/full/test --conf 0.4 --out results/inference
"""

import argparse
import csv
import json
import time
from pathlib import Path

import numpy as np
from PIL import Image, ImageDraw

Image.MAX_IMAGE_PIXELS = None


def read_rgb(path):
    path = Path(path)
    if path.suffix.lower() in (".tif", ".tiff"):
        try:
            import tifffile
            arr = tifffile.imread(str(path))
            if arr.ndim == 2:
                arr = np.stack([arr] * 3, -1)
            if arr.shape[0] in (3, 4) and arr.shape[-1] not in (3, 4):
                arr = np.moveaxis(arr, 0, -1)
            return np.ascontiguousarray(arr[..., :3]).astype(np.uint8)
        except Exception:
            pass
    return np.array(Image.open(path).convert("RGB"))


def tile_starts(length, tile, stride):
    if length <= tile:
        return [0]
    s = list(range(0, length - tile + 1, stride))
    if s[-1] != length - tile:
        s.append(length - tile)
    return s


def center_nms(dets, radius):
    """dets: (N, 5) [x1, y1, x2, y2, score]. Keep the highest-scoring box and
    drop any other box whose center is within `radius` px (duplicates from
    overlapping tiles, or two boxes on the same cell)."""
    if len(dets) == 0:
        return dets
    dets = dets[np.argsort(-dets[:, 4])]
    centers = np.stack([(dets[:, 0] + dets[:, 2]) / 2, (dets[:, 1] + dets[:, 3]) / 2], 1)
    keep, removed = [], np.zeros(len(dets), bool)
    for i in range(len(dets)):
        if removed[i]:
            continue
        keep.append(i)
        d = np.linalg.norm(centers - centers[i], axis=1)
        removed |= d < radius
    return dets[keep]


def predict_large_image(model, arr, tile=640, overlap=64, conf=0.05, batch=16,
                        nms_radius=25, device=None, bg_thresh=225):
    """Returns (N, 5) array of detections in full-image pixel coordinates."""
    h, w = arr.shape[:2]
    stride = tile - overlap
    coords, crops, all_dets = [], [], []

    def flush():
        if not crops:
            return
        results = model.predict(crops, imgsz=tile, conf=conf, device=device, verbose=False)
        for (x0, y0), r in zip(coords, results):
            b = r.boxes
            if len(b) == 0:
                continue
            xyxy = b.xyxy.cpu().numpy()
            sc = b.conf.cpu().numpy()[:, None]
            xyxy[:, [0, 2]] += x0
            xyxy[:, [1, 3]] += y0
            all_dets.append(np.hstack([xyxy, sc]))
        coords.clear()
        crops.clear()

    n_tiles = n_skipped = 0
    for y0 in tile_starts(h, tile, stride):
        for x0 in tile_starts(w, tile, stride):
            crop = arr[y0:y0 + tile, x0:x0 + tile]
            n_tiles += 1
            if crop.mean() > bg_thresh:
                n_skipped += 1
                continue
            if crop.shape[:2] != (tile, tile):
                pad = np.full((tile, tile, 3), 255, np.uint8)
                pad[:crop.shape[0], :crop.shape[1]] = crop
                crop = pad
            crops.append(np.ascontiguousarray(crop[..., ::-1]))  # Ultralytics expects BGR arrays
            coords.append((x0, y0))
            if len(crops) == batch:
                flush()
    flush()
    dets = np.vstack(all_dets) if all_dets else np.zeros((0, 5))
    return center_nms(dets, nms_radius), {"tiles": n_tiles, "background_skipped": n_skipped}


def density_heatmap(dets, shape, cell=512):
    h, w = shape[:2]
    grid = np.zeros((int(np.ceil(h / cell)), int(np.ceil(w / cell))))
    for x1, y1, x2, y2, _ in dets:
        cy = min(int((y1 + y2) / 2 // cell), grid.shape[0] - 1)
        cx = min(int((x1 + x2) / 2 // cell), grid.shape[1] - 1)
        grid[cy, cx] += 1
    return grid


def save_overview(arr, dets, path, max_side=2400, gt=None):
    """Downscaled image with predictions (red circles) and optional ground truth (green)."""
    h, w = arr.shape[:2]
    s = min(1.0, max_side / max(h, w))
    img = Image.fromarray(arr).resize((int(w * s), int(h * s)))
    draw = ImageDraw.Draw(img)
    r = max(6, int(35 * s))
    for x1, y1, x2, y2 in (gt if gt is not None else []):
        cx, cy = (x1 + x2) / 2 * s, (y1 + y2) / 2 * s
        draw.ellipse([cx - r - 3, cy - r - 3, cx + r + 3, cy + r + 3], outline=(0, 200, 0), width=3)
    for x1, y1, x2, y2, sc in dets:
        cx, cy = (x1 + x2) / 2 * s, (y1 + y2) / 2 * s
        draw.ellipse([cx - r, cy - r, cx + r, cy + r], outline=(230, 0, 0), width=3)
    img.save(path, quality=90)


def save_heatmap(grid, path, title):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(grid, cmap="magma", interpolation="nearest")
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, label="mitoses per 512 x 512 px region")
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--source", required=True, help="Image file or folder of TIFF/JPG/PNG")
    ap.add_argument("--out", default="results/inference")
    ap.add_argument("--conf", type=float, default=0.4, help="Use the best threshold found by evaluate_midog.py")
    ap.add_argument("--tile", type=int, default=640)
    ap.add_argument("--overlap", type=int, default=64)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--device", default=None)
    args = ap.parse_args()

    from ultralytics import YOLO
    model = YOLO(args.weights)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    src = Path(args.source)
    exts = {".tif", ".tiff", ".jpg", ".jpeg", ".png"}
    files = sorted(p for p in src.iterdir() if p.suffix.lower() in exts) if src.is_dir() else [src]

    summary = []
    for p in files:
        t0 = time.time()
        arr = read_rgb(p)
        dets, info = predict_large_image(model, arr, args.tile, args.overlap, args.conf,
                                         args.batch, device=args.device)
        dt = time.time() - t0
        mp = arr.shape[0] * arr.shape[1] / 1e6
        with open(out / f"{p.stem}_detections.csv", "w", newline="") as f:
            wr = csv.writer(f)
            wr.writerow(["x1", "y1", "x2", "y2", "confidence"])
            wr.writerows(np.round(dets, 3).tolist())
        save_overview(arr, dets, out / f"{p.stem}_overview.jpg")
        save_heatmap(density_heatmap(dets, arr.shape), out / f"{p.stem}_heatmap.png",
                     f"{p.name}: {len(dets)} mitotic figures")
        row = {"image": p.name, "width": arr.shape[1], "height": arr.shape[0],
               "mitotic_count": int(len(dets)), "seconds": round(dt, 2),
               "megapixels_per_second": round(mp / dt, 2), **info}
        summary.append(row)
        print(f"{p.name}: {len(dets)} mitoses | {arr.shape[1]}x{arr.shape[0]} px | "
              f"{dt:.1f}s ({mp / dt:.1f} MP/s) | {info['background_skipped']}/{info['tiles']} tiles skipped")

    (out / "summary.json").write_text(json.dumps(summary, indent=2))
    print(f"\nResults written to {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/evaluate_midog.py
"""
Evaluate the mitosis detector the way the MIDOG challenge does: on FULL
images, with detection F1 score.

Steps
  1. Tiled inference on every full val and test image (low confidence, once).
  2. Match predictions to ground-truth mitoses: a prediction is a true positive
     if its center is within 30 px (about 7.5 micrometres, the MIDOG criterion)
     of an unmatched ground-truth mitosis. Greedy, highest confidence first.
  3. Pick the confidence threshold with the best F1 on VAL, then report TEST
     at that threshold (no tuning on test).
  4. Report overall and per-tumor-type precision / recall / F1, mitotic count
     error and count correlation, plus tile-level mAP from Ultralytics.

Usage
  python evaluate_midog.py --weights runs/midog/midog_yolov8s/weights/best.pt \
      --data-dir datasets/midog_yolo
"""

import argparse
import csv
import json
from collections import defaultdict
from pathlib import Path

import numpy as np

from tiled_inference import predict_large_image, read_rgb, save_overview


def match(dets, gt_centers, radius):
    """Returns (tp, fp, fn) for detections already filtered by threshold."""
    if len(gt_centers) == 0:
        return 0, len(dets), 0
    if len(dets) == 0:
        return 0, 0, len(gt_centers)
    dets = dets[np.argsort(-dets[:, 4])]
    centers = np.stack([(dets[:, 0] + dets[:, 2]) / 2, (dets[:, 1] + dets[:, 3]) / 2], 1)
    used = np.zeros(len(gt_centers), bool)
    tp = 0
    for c in centers:
        d = np.linalg.norm(gt_centers - c, axis=1)
        d[used] = np.inf
        j = int(np.argmin(d))
        if d[j] <= radius:
            used[j] = True
            tp += 1
    return tp, len(dets) - tp, len(gt_centers) - tp


def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f


def scores_at(cases, thr, radius):
    tot = np.zeros(3, int)
    per_type = defaultdict(lambda: np.zeros(3, int))
    for c in cases:
        d = c["dets"][c["dets"][:, 4] >= thr]
        m = np.array(match(d, c["gt"], radius))
        tot += m
        per_type[c["tumor"]] += m
    return tot, per_type


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True)
    ap.add_argument("--data-dir", type=Path, default=Path("datasets/midog_yolo"))
    ap.add_argument("--ann", type=Path, default=None,
                    help="MIDOG++.json (default: <data-dir>/MIDOG++.json, copied there by Session A)")
    ap.add_argument("--out", type=Path, default=Path("results/evaluation"))
    ap.add_argument("--radius", type=float, default=30.0, help="Match distance in px (~7.5 um)")
    ap.add_argument("--tile", type=int, default=640)
    ap.add_argument("--overlap", type=int, default=64)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--device", default=None)
    ap.add_argument("--skip-map", action="store_true", help="Skip the tile-level mAP evaluation")
    args = ap.parse_args()

    from ultralytics import YOLO
    model = YOLO(args.weights)
    args.out.mkdir(parents=True, exist_ok=True)
    (args.out / "overviews").mkdir(exist_ok=True)

    ann = json.loads((args.ann or args.data_dir / "MIDOG++.json").read_text())
    gt_boxes = defaultdict(list)
    for a in ann["annotations"]:
        if a["category_id"] == 1:
            gt_boxes[a["image_id"]].append(a["bbox"])

    with open(args.data_dir / "split.csv") as f:
        split_rows = list(csv.DictReader(f))

    cases = {"val": [], "test": []}
    for row in split_rows:
        if row["split"] not in cases:
            continue
        path = args.data_dir / "full" / row["split"] / row["file_name"]
        if not path.exists():
            print(f"  ! missing {path}, skipped")
            continue
        arr = read_rgb(path)
        dets, _ = predict_large_image(model, arr, args.tile, args.overlap, conf=0.05,
                                      batch=args.batch, device=args.device)
        boxes = np.array(gt_boxes[int(row["image_id"])], float).reshape(-1, 4)
        gt = np.stack([(boxes[:, 0] + boxes[:, 2]) / 2, (boxes[:, 1] + boxes[:, 3]) / 2], 1) if len(boxes) else np.zeros((0, 2))
        cases[row["split"]].append({"name": row["file_name"], "tumor": row["tumor_type"],
                                    "dets": dets, "gt": gt, "gt_boxes": boxes, "arr_path": path})
        print(f"[{row['split']}] {row['file_name']} ({row['tumor_type']}): "
              f"{len(gt)} true mitoses, {int((dets[:, 4] >= 0.3).sum())} predicted at conf 0.3")

    # 1) threshold selection on val
    thresholds = np.round(np.arange(0.05, 0.96, 0.025), 3)
    sweep = []
    tune_on = "val" if cases["val"] else "test"
    for t in thresholds:
        tot, _ = scores_at(cases[tune_on], t, args.radius)
        sweep.append((t, *prf(*tot)))
    best_t, _, _, best_f1_val = max(sweep, key=lambda x: x[3])
    print(f"\nBest threshold on {tune_on}: {best_t:.3f} (F1 {best_f1_val:.3f})")

    # 2) report test at that threshold
    report = {"threshold": float(best_t), "tuned_on": tune_on, "match_radius_px": args.radius}
    for split in ("val", "test"):
        if not cases[split]:
            continue
        tot, per_type = scores_at(cases[split], best_t, args.radius)
        p, r, f = prf(*tot)
        true_counts = [len(c["gt"]) for c in cases[split]]
        pred_counts = [int((c["dets"][:, 4] >= best_t).sum()) for c in cases[split]]
        corr = float(np.corrcoef(true_counts, pred_counts)[0, 1]) if len(set(true_counts)) > 1 and len(set(pred_counts)) > 1 else None
        report[split] = {
            "images": len(cases[split]),
            "tp": int(tot[0]), "fp": int(tot[1]), "fn": int(tot[2]),
            "precision": round(p, 4), "recall": round(r, 4), "f1": round(f, 4),
            "mean_abs_count_error": round(float(np.mean(np.abs(np.array(true_counts) - np.array(pred_counts)))), 2),
            "count_correlation": None if corr is None else round(corr, 4),
            "per_tumor_type": {t: dict(zip(["precision", "recall", "f1"], [round(x, 4) for x in prf(*m)]),
                                       tp=int(m[0]), fp=int(m[1]), fn=int(m[2]))
                               for t, m in sorted(per_type.items())},
        }

    # 3) per-image counts + overview images for the test set
    with open(args.out / "per_image_counts.csv", "w", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(["split", "image", "tumor_type", "true_count", "predicted_count"])
        for split in ("val", "test"):
            for c in cases[split]:
                wr.writerow([split, c["name"], c["tumor"], len(c["gt"]), int((c["dets"][:, 4] >= best_t).sum())])
    for c in cases["test"][:8]:
        save_overview(read_rgb(c["arr_path"]), c["dets"][c["dets"][:, 4] >= best_t],
                      args.out / "overviews" / f"{Path(c['name']).stem}.jpg", gt=c["gt_boxes"])

    # 4) threshold curve plot
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    sw = np.array(sweep)
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for i, lab in [(1, "precision"), (2, "recall"), (3, "F1")]:
        ax.plot(sw[:, 0], sw[:, i], label=lab, lw=2)
    ax.axvline(best_t, ls="--", c="gray")
    ax.set_xlabel("confidence threshold"); ax.set_ylabel("score"); ax.set_ylim(0, 1)
    ax.set_title(f"Full-image detection on {tune_on} (best F1 {best_f1_val:.3f} at {best_t:.2f})")
    ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
    fig.savefig(args.out / "threshold_curve.png", dpi=150); plt.close(fig)

    # 5) standard tile-level mAP on the test tiles
    if not args.skip_map:
        m = model.val(data=str(args.data_dir / "data.yaml"), split="test", imgsz=args.tile,
                      batch=args.batch, device=args.device, plots=True, verbose=False,
                      project=str(args.out), name="tile_map", exist_ok=True)
        report["test_tiles_map"] = {"mAP50": round(float(m.box.map50), 4), "mAP50-95": round(float(m.box.map), 4),
                                    "precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4)}

    (args.out / "metrics.json").write_text(json.dumps(report, indent=2))
    print("\n" + json.dumps({k: v for k, v in report.items() if k != "val"}, indent=2))
    print(f"\nAll results in {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/train_midog.py
"""
Fine-tune YOLOv8 for mitotic figure detection on the tiled MIDOG++ dataset.

Augmentations are applied on the fly by Ultralytics (nothing extra is saved
to disk). They are tuned for histopathology:
  * vertical + horizontal flips: tissue has no "up", so both are always valid
  * strong HSV jitter: simulates stain and scanner colour variation, the main
    source of domain shift between labs
  * mild scale + mosaic; no shear/perspective (would distort cell shape)

Usage
  python train_midog.py --data datasets/midog_yolo/data.yaml
  python train_midog.py --data ... --epochs 40 --batch 16 --model yolov8s.pt
"""

import argparse
import time
from pathlib import Path

import torch
from ultralytics import YOLO


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--model", default="yolov8s.pt", help="Pretrained weights to start from")
    ap.add_argument("--epochs", type=int, default=40)
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--batch", type=int, default=16, help="Lower to 8 if you run out of GPU memory")
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--patience", type=int, default=12)
    ap.add_argument("--name", default="midog_yolov8s")
    ap.add_argument("--device", default=None, help="0 for first GPU, cpu for CPU")
    ap.add_argument("--hours", type=float, default=None,
                    help="Safety cap: stop after this many hours (does not change the epoch schedule)")
    ap.add_argument("--project", default="runs/midog")
    ap.add_argument("--cache", default="ram", help="ram / disk / none: caching decoded tiles speeds up epochs")
    args = ap.parse_args()

    device = args.device if args.device is not None else (0 if torch.cuda.is_available() else "cpu")
    if device == "cpu":
        print("WARNING: no CUDA GPU found, training on CPU will be very slow.")
    else:
        print(f"Training on GPU: {torch.cuda.get_device_name(0)}")

    model = YOLO(args.model)
    t0 = time.time()
    if args.hours:
        # Safety cap that only STOPS training. Ultralytics' own `time=` argument
        # re-plans the number of epochs from the budget, which stretches the
        # learning-rate schedule (with a generous budget the LR barely decays).
        def stop_after_budget(trainer):
            if time.time() - t0 > args.hours * 3600:
                print(f"\nTime budget of {args.hours} h reached, stopping after epoch {trainer.epoch + 1}.")
                trainer.stop = True
        model.add_callback("on_fit_epoch_end", stop_after_budget)
    model.train(
        data=args.data,
        epochs=args.epochs,
        imgsz=args.imgsz,
        batch=args.batch,
        workers=args.workers,
        device=device,
        project=args.project,
        name=args.name,
        exist_ok=True,
        cache=False if args.cache == "none" else args.cache,
        optimizer="AdamW",
        lr0=0.001,
        cos_lr=True,
        warmup_epochs=2,
        amp=True,
        patience=args.patience,
        # histopathology augmentations
        fliplr=0.5,
        flipud=0.5,
        degrees=0.0,
        hsv_h=0.04,
        hsv_s=0.5,
        hsv_v=0.3,
        scale=0.2,
        translate=0.1,
        shear=0.0,
        perspective=0.0,
        mosaic=1.0,
        close_mosaic=5,
        mixup=0.0,
        plots=True,
        verbose=True,
    )
    print(f"\nTraining time: {(time.time() - t0) / 60:.1f} min")
    save_dir = Path(model.trainer.save_dir).resolve()
    best = save_dir / "weights" / "best.pt"
    Path("last_run.txt").write_text(str(save_dir))  # lets the notebook find the run reliably
    print(f"Run folder: {save_dir}")
    print(f"Best weights: {best}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/code/two_stage.py
"""
Two-stage mitosis detection: YOLOv8 candidates -> EfficientNet-B0 verification.

Stage 1 (detector) is run at a low threshold so it finds almost every mitosis.
Stage 2 (classifier) looks at a small crop around each candidate and decides
"real mitosis" vs "look-alike". This follows the detect-then-refine design of
the TIA Centre MIDOG 2021 entry (Jahanifar et al., arXiv:2109.00853).

Steps
  1. Hard negative mining: run the detector on the TRAIN tiles and keep its
     false positives (no true mitosis within 30 px) as extra negatives.
  2. Train the classifier on train crops: mitoses (positive) vs imposters +
     mined false positives (negative). Model selection on VAL candidates.
  3. Evaluate on full val/test images. Score variants: detector only,
     classifier only, geometric mean, average. The variant AND threshold are
     chosen on VAL only; TEST is reported once at that setting.

Usage
  python two_stage.py --weights runs/.../best.pt --data-dir datasets/midog_yolo --out results/two_stage
"""

import argparse
import csv
import json
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image

from evaluate_midog import match, prf
from tiled_inference import predict_large_image, read_rgb, save_overview

CROP = 128          # saved crop size (matches prepare_midog.py --crop-size)
INNER = 112         # random/center crop fed to the network (allows +-8 px jitter)
NET_SIZE = int(__import__("os").environ.get("NET_SIZE", 224))      # upsampled input size for the ImageNet-pretrained network
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def crop_centered(arr, cx, cy, size=CROP):
    h, w = arr.shape[:2]
    x0, y0 = int(round(cx)) - size // 2, int(round(cy)) - size // 2
    out = np.full((size, size, 3), 255, np.uint8)
    sx0, sy0, sx1, sy1 = max(0, x0), max(0, y0), min(w, x0 + size), min(h, y0 + size)
    if sx1 > sx0 and sy1 > sy0:
        out[sy0 - y0:sy1 - y0, sx0 - x0:sx1 - x0] = arr[sy0:sy1, sx0:sx1]
    return out


def gt_centers_by_image(ann_path):
    ann = json.loads(Path(ann_path).read_text())
    by_name = {im["file_name"]: im["id"] for im in ann["images"]}
    centers = defaultdict(list)
    for a in ann["annotations"]:
        if a["category_id"] == 1:
            x1, y1, x2, y2 = a["bbox"]
            centers[a["image_id"]].append(((x1 + x2) / 2, (y1 + y2) / 2))
    return by_name, {k: np.array(v, float) for k, v in centers.items()}


# --------------------------------------------------------------------------- #
# 1. Hard negative mining on training tiles
# --------------------------------------------------------------------------- #
def mine_hard_negatives(model, data_dir, ann_path, out_dir, conf=0.1, radius=30, batch=32,
                        max_ratio=1.5, device=None):
    by_name, gt = gt_centers_by_image(ann_path)
    tiles = sorted((data_dir / "images" / "train").glob("*.jpg"))
    n_pos = len(list((data_dir / "crops" / "train" / "mitosis").glob("*.jpg")))
    cands = []  # (score, stem, gx, gy, tile_path, lx, ly)
    for i in range(0, len(tiles), batch):
        chunk = tiles[i:i + batch]
        results = model.predict([str(p) for p in chunk], imgsz=640, conf=conf, device=device, verbose=False)
        for p, r in zip(chunk, results):
            if len(r.boxes) == 0:
                continue
            stem, xs, ys = p.stem.rsplit("_", 2)
            x0, y0 = int(xs[1:]), int(ys[1:])
            centers = gt.get(by_name.get(stem + ".tiff"), np.zeros((0, 2)))
            xyxy = r.boxes.xyxy.cpu().numpy()
            sc = r.boxes.conf.cpu().numpy()
            for (bx1, by1, bx2, by2), s in zip(xyxy, sc):
                lx, ly = (bx1 + bx2) / 2, (by1 + by2) / 2
                gx, gy = lx + x0, ly + y0
                if len(centers) and np.min(np.hypot(centers[:, 0] - gx, centers[:, 1] - gy)) <= radius:
                    continue  # true positive, not a hard negative
                cands.append((float(s), stem, gx, gy, p, lx, ly))
    # de-duplicate the same false positive seen in overlapping tiles, keep the hardest
    cands.sort(key=lambda c: -c[0])
    seen, kept = set(), []
    for c in cands:
        key = (c[1], int(c[2] // 20), int(c[3] // 20))
        if key in seen:
            continue
        seen.add(key)
        kept.append(c)
    kept = kept[: int(max_ratio * n_pos)] if n_pos else kept
    out_dir.mkdir(parents=True, exist_ok=True)
    cache = {}
    for k, (s, stem, gx, gy, p, lx, ly) in enumerate(kept):
        if p not in cache:
            cache = {p: np.array(Image.open(p).convert("RGB"))}
        Image.fromarray(crop_centered(cache[p], lx, ly)).save(out_dir / f"{stem}_{int(gx)}_{int(gy)}.jpg", quality=95)
    print(f"Hard negative mining: {len(cands)} false positives on {len(tiles)} train tiles, "
          f"{len(kept)} unique kept (cap {max_ratio}x the {n_pos} positives)")
    return len(kept)


# --------------------------------------------------------------------------- #
# 2. Classifier
# --------------------------------------------------------------------------- #
def to_tensor(img_np):
    t = torch.from_numpy(np.ascontiguousarray(img_np)).permute(2, 0, 1).float() / 255.0
    return t


class CropDataset(torch.utils.data.Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train  # items: list of (path or array, label)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        src, y = self.items[i]
        a = np.array(Image.open(src).convert("RGB")) if isinstance(src, (str, Path)) else src
        if self.train:
            ox, oy = np.random.randint(0, CROP - INNER + 1, 2)
            a = a[oy:oy + INNER, ox:ox + INNER]
            a = np.rot90(a, np.random.randint(4))
            if np.random.rand() < 0.5:
                a = a[:, ::-1]
        else:
            o = (CROP - INNER) // 2
            a = a[o:o + INNER, o:o + INNER]
        t = to_tensor(a)
        if self.train:  # stain / colour jitter
            t = t * (1 + 0.2 * (torch.rand(3, 1, 1) - 0.5)) + 0.1 * (torch.rand(3, 1, 1) - 0.5)
            t = t.clamp(0, 1)
        t = nn.functional.interpolate(t[None], size=NET_SIZE, mode="bilinear", align_corners=False)[0]
        return (t - MEAN) / STD, torch.tensor(y, dtype=torch.long)


def build_model(pretrained=True):
    from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    return m


@torch.no_grad()
def predict_probs(net, crops, device, batch=256, tta=True):
    """crops: list of CROP x CROP uint8 arrays. Returns P(mitosis) with 8-fold
    flip/rotation test-time augmentation."""
    net.eval()
    ds = CropDataset([(c, 0) for c in crops], train=False)
    probs = []
    for i in range(0, len(ds), batch):
        x = torch.stack([ds[j][0] for j in range(i, min(i + batch, len(ds)))]).to(device)
        views = [x, x.flip(3)] if not tta else [torch.rot90(v, k, (2, 3)) for v in (x, x.flip(3)) for k in range(4)]
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            p = torch.stack([net(v).float().softmax(1)[:, 1] for v in views]).mean(0)
        probs.append(p.cpu())
    return torch.cat(probs).numpy() if probs else np.zeros(0)


def roc_auc(y, s):
    """Tie-aware ROC AUC (average ranks), no sklearn needed."""
    y, s = np.asarray(y), np.asarray(s, float)
    n1, n0 = int((y == 1).sum()), int((y == 0).sum())
    if n1 == 0 or n0 == 0:
        return float("nan")
    order = np.argsort(s, kind="mergesort")
    ranks = np.empty(len(s))
    sorted_s = s[order]
    i = 0
    while i < len(s):
        j = i
        while j + 1 < len(s) and sorted_s[j + 1] == sorted_s[i]:
            j += 1
        ranks[order[i:j + 1]] = (i + j) / 2 + 1  # average rank for ties
        i = j + 1
    return float((ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def train_classifier(train_items, val_crops, val_labels, device, epochs=8, batch=128, lr=3e-4,
                     workers=4, out_path=None, pretrained=True):
    net = build_model(pretrained).to(device)
    n1 = sum(1 for _, y in train_items if y == 1)
    n0 = len(train_items) - n1
    w = torch.tensor([len(train_items) / (2 * max(n0, 1)), len(train_items) / (2 * max(n1, 1))], device=device)
    loss_fn = nn.CrossEntropyLoss(weight=w, label_smoothing=0.05)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
    dl = torch.utils.data.DataLoader(CropDataset(train_items, True), batch_size=batch, shuffle=True,
                                     num_workers=workers, drop_last=len(train_items) > batch,
                                     pin_memory=device.type == "cuda",
                                     worker_init_fn=lambda i: np.random.seed((torch.initial_seed() + i) % 2**32))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=max(1, epochs * len(dl)), pct_start=0.15)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    best_auc, best_state, history = -1.0, None, []
    print(f"Classifier: {n1} positive / {n0} negative train crops, {len(val_crops)} val candidates")
    for ep in range(epochs):
        net.train()
        t0, tot = time.time(), 0.0
        for x, y in dl:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                loss = loss_fn(net(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            tot += loss.item() * len(y)
        p = predict_probs(net, val_crops, device, tta=False) if len(val_crops) else np.zeros(0)
        auc = roc_auc(val_labels, p) if len(val_crops) else float("nan")
        history.append({"epoch": ep + 1, "train_loss": tot / len(train_items), "val_auc": auc})
        print(f"  epoch {ep + 1}/{epochs}  loss {tot / len(train_items):.4f}  val AUC {auc:.4f}  ({time.time() - t0:.0f}s)")
        if not np.isnan(auc) and auc > best_auc or best_state is None:
            best_auc = auc
            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
    net.load_state_dict(best_state)
    if out_path:
        torch.save(best_state, out_path)
    return net, history, best_auc


# --------------------------------------------------------------------------- #
# 3. Two-stage evaluation
# --------------------------------------------------------------------------- #
VARIANTS = {
    "detector only": lambda d, p: d,
    "classifier only": lambda d, p: p,
    "geometric mean": lambda d, p: np.sqrt(d * p),
    "average": lambda d, p: (d + p) / 2,
}


def evaluate_variant(cases, variant, thr, radius):
    tot = np.zeros(3, int)
    per_type = defaultdict(lambda: np.zeros(3, int))
    true_c, pred_c = [], []
    for c in cases:
        s = VARIANTS[variant](c["dets"][:, 4], c["probs"]) if len(c["dets"]) else np.zeros(0)
        d = c["dets"][s >= thr] if len(c["dets"]) else c["dets"]
        m = np.array(match(d, c["gt"], radius))
        tot += m
        per_type[c["tumor"]] += m
        true_c.append(len(c["gt"]))
        pred_c.append(len(d))
    return tot, per_type, true_c, pred_c


def summarise(tot, per_type, true_c, pred_c):
    p, r, f = prf(*tot)
    tc, pc = np.array(true_c), np.array(pred_c)
    corr = float(np.corrcoef(tc, pc)[0, 1]) if len(set(tc)) > 1 and len(set(pc)) > 1 else None
    return {"tp": int(tot[0]), "fp": int(tot[1]), "fn": int(tot[2]),
            "precision": round(p, 4), "recall": round(r, 4), "f1": round(f, 4),
            "mean_abs_count_error": round(float(np.mean(np.abs(tc - pc))), 2),
            "count_correlation": None if corr is None else round(corr, 4),
            "per_tumor_type": {t: dict(zip(["precision", "recall", "f1"], [round(x, 4) for x in prf(*m)]),
                                       tp=int(m[0]), fp=int(m[1]), fn=int(m[2])) for t, m in sorted(per_type.items())}}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", required=True, help="Trained YOLOv8 detector (best.pt)")
    ap.add_argument("--data-dir", type=Path, required=True)
    ap.add_argument("--ann", type=Path, default=None, help="MIDOG++.json (default: <data-dir>/MIDOG++.json)")
    ap.add_argument("--out", type=Path, default=Path("results/two_stage"))
    ap.add_argument("--epochs", type=int, default=8)
    ap.add_argument("--batch", type=int, default=128)
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--cand-conf", type=float, default=0.05, help="Stage-1 threshold for candidates")
    ap.add_argument("--radius", type=float, default=30.0)
    ap.add_argument("--no-pretrained", action="store_true", help="(testing only) skip ImageNet weights")
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args()

    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ann_path = args.ann or args.data_dir / "MIDOG++.json"
    args.out.mkdir(parents=True, exist_ok=True)
    from ultralytics import YOLO
    det = YOLO(args.weights)
    t_start = time.time()

    # --- 1. hard negative mining
    mined_dir = args.data_dir / "crops" / "train" / "mined"
    n_mined = mine_hard_negatives(det, args.data_dir, ann_path, mined_dir, device=None if device.type == "cuda" else "cpu")

    # --- candidates on full val / test images (stage 1)
    by_name, gt = gt_centers_by_image(ann_path)
    with open(args.data_dir / "split.csv") as f:
        rows = [r for r in csv.DictReader(f) if r["split"] in ("val", "test")]
    cases = {"val": [], "test": []}
    for r in rows:
        path = args.data_dir / "full" / r["split"] / r["file_name"]
        if not path.exists():
            print(f"  ! missing {path}")
            continue
        arr = read_rgb(path)
        dets, _ = predict_large_image(det, arr, conf=args.cand_conf, batch=16,
                                      device=None if device.type == "cuda" else "cpu")
        crops = [crop_centered(arr, (x1 + x2) / 2, (y1 + y2) / 2) for x1, y1, x2, y2, _ in dets]
        g = gt.get(int(r["image_id"]), np.zeros((0, 2)))
        # label each candidate for classifier validation: 1 if a true mitosis is within the radius
        lab = [int(len(g) > 0 and np.min(np.hypot(g[:, 0] - (x1 + x2) / 2, g[:, 1] - (y1 + y2) / 2)) <= args.radius)
               for x1, y1, x2, y2, _ in dets]
        cases[r["split"]].append({"name": r["file_name"], "tumor": r["tumor_type"], "dets": dets,
                                  "crops": crops, "cand_labels": lab, "gt": g, "path": path})
        print(f"[{r['split']}] {r['file_name']}: {len(dets)} candidates, {len(g)} true mitoses")

    # --- 2. classifier
    cdir = args.data_dir / "crops" / "train"
    train_items = [(p, 1) for p in (cdir / "mitosis").glob("*.jpg")]
    train_items += [(p, 0) for p in (cdir / "imposter").glob("*.jpg")]
    train_items += [(p, 0) for p in mined_dir.glob("*.jpg")]
    val_crops = [c for case in cases["val"] for c in case["crops"]]
    val_labels = [l for case in cases["val"] for l in case["cand_labels"]]
    net, history, best_auc = train_classifier(train_items, val_crops, val_labels, device, epochs=args.epochs,
                                              batch=args.batch, workers=args.workers,
                                              out_path=args.out / "classifier_effnet_b0.pt",
                                              pretrained=not args.no_pretrained)

    # --- 3. score every candidate, choose variant + threshold on val, report test
    for split in ("val", "test"):
        for c in cases[split]:
            c["probs"] = predict_probs(net, c["crops"], device) if c["crops"] else np.zeros(0)

    thresholds = np.round(np.arange(0.05, 0.96, 0.025), 3)
    sweep = {}
    for v in VARIANTS:
        best = max(((t, prf(*evaluate_variant(cases["val"], v, t, args.radius)[0])[2]) for t in thresholds),
                   key=lambda x: x[1])
        sweep[v] = {"threshold": float(best[0]), "val_f1": round(best[1], 4)}
    chosen = max((v for v in VARIANTS if v != "detector only"), key=lambda v: sweep[v]["val_f1"])
    final = chosen if sweep[chosen]["val_f1"] > sweep["detector only"]["val_f1"] else "detector only"

    report = {"classifier": {"train_crops": len(train_items), "mined_hard_negatives": n_mined,
                             "best_val_auc": round(best_auc, 4), "history": history},
              "selection_on_val": sweep, "chosen_variant": final, "variants": {}}
    for v in VARIANTS:
        t = sweep[v]["threshold"]
        report["variants"][v] = {"threshold": t,
                                 "val": summarise(*evaluate_variant(cases["val"], v, t, args.radius)),
                                 "test": summarise(*evaluate_variant(cases["test"], v, t, args.radius))}
    report["final"] = {"variant": final, **report["variants"][final]}
    report["minutes"] = round((time.time() - t_start) / 60, 1)

    # per-image counts and overviews for the final model
    t = sweep[final]["threshold"]
    with open(args.out / "per_image_counts.csv", "w", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(["split", "image", "tumor_type", "true_count", "predicted_count"])
        for split in ("val", "test"):
            for c in cases[split]:
                s = VARIANTS[final](c["dets"][:, 4], c["probs"]) if len(c["dets"]) else np.zeros(0)
                wr.writerow([split, c["name"], c["tumor"], len(c["gt"]), int((s >= t).sum())])
    (args.out / "overviews").mkdir(exist_ok=True)
    ann = json.loads(Path(ann_path).read_text())
    for c in cases["test"][:8]:
        s = VARIANTS[final](c["dets"][:, 4], c["probs"]) if len(c["dets"]) else np.zeros(0)
        gtb = np.array([a["bbox"] for a in ann["annotations"]
                        if a["image_id"] == by_name[c["name"]] and a["category_id"] == 1], float).reshape(-1, 4)
        save_overview(read_rgb(c["path"]), c["dets"][s >= t], args.out / "overviews" / f"{Path(c['name']).stem}.jpg", gt=gtb)

    (args.out / "metrics.json").write_text(json.dumps(report, indent=2))
    print("\nSelection on val:", json.dumps(sweep, indent=2))
    print(f"Chosen: {final}")
    for v in VARIANTS:
        m = report["variants"][v]["test"]
        print(f"  TEST {v:<16} P {m['precision']:.3f}  R {m['recall']:.3f}  F1 {m['f1']:.3f}  "
              f"count r {m['count_correlation']}")
    print(f"\nResults in {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
# 1) Retrain the detector on the bigger dataset (about 40 to 50 minutes)
%cd {CODE}
!python train_midog.py --data {DATA}/data.yaml --model {MODEL} --epochs {EPOCHS} --hours {HOURS} --patience {PATIENCE} --batch {BATCH} --workers 4 --project {WORK}/runs --name midog_yolov8s_v2
RUN = Path(open(f'{CODE}/last_run.txt').read().strip())
BEST = RUN / 'weights' / 'best.pt'
print('Detector weights:', BEST, BEST.exists())

In [ ]:
# 2) Detector-only evaluation (same protocol as Session B) + tile-level mAP
!python evaluate_midog.py --weights {BEST} --data-dir {DATA} --out {WORK}/results/detector_only --batch {BATCH}

In [ ]:
# 3) Two-stage: hard negative mining -> EfficientNet-B0 classifier -> fused scoring (about 10 to 15 minutes)
!python two_stage.py --weights {BEST} --data-dir {DATA} --out {WORK}/results/two_stage --epochs {CLS_EPOCHS} --workers 4

In [ ]:
# 4) Comparison table for README / resume
from IPython.display import Image as Show, display
det = json.load(open(f'{WORK}/results/detector_only/metrics.json'))
ts = json.load(open(f'{WORK}/results/two_stage/metrics.json'))
man = json.load(open(f'{DATA}/manifest.json'))['splits']
fin = ts['final']
best2 = max((v for v in ts['variants'] if v != 'detector only'), key=lambda v: ts['selection_on_val'][v]['val_f1'])

def row(name, m):
    return f"| {name} | {m['precision']:.3f} | {m['recall']:.3f} | **{m['f1']:.3f}** | {m['count_correlation']} |"

lines = ['# MIDOG++ mitosis detection: results (same 14 test images in every row)', '',
         '| Model | Precision | Recall | F1 | Count correlation |', '|---|---|---|---|---|',
         row(BASELINE['name'], BASELINE),
         row(f"Bigger training set ({man['train']['images']} train images), detector only", det['test']),
         row(f'Two-stage: detector + EfficientNet-B0 ({best2})', ts['variants'][best2]['test']),
         '', f"Final model (chosen on val): **{fin['variant']}**, threshold {fin['threshold']}", '',
         f"Classifier: {ts['classifier']['train_crops']} train crops incl. {ts['classifier']['mined_hard_negatives']} mined hard negatives, "
         f"best val AUC {ts['classifier']['best_val_auc']}", '',
         'Score variants on test (each with its own val-tuned threshold):', '',
         '| Variant | Val F1 | Test P | Test R | Test F1 |', '|---|---|---|---|---|']
for v, d in ts['variants'].items():
    lines.append(f"| {v} | {ts['selection_on_val'][v]['val_f1']:.3f} | {d['test']['precision']:.3f} | {d['test']['recall']:.3f} | {d['test']['f1']:.3f} |")
lines += ['', '## Final model per tumor type (test)', '', '| Tumor type | Precision | Recall | F1 |', '|---|---|---|---|']
for k, v in fin['test']['per_tumor_type'].items():
    lines.append(f"| {k} | {v['precision']:.3f} | {v['recall']:.3f} | {v['f1']:.3f} |")
lines += ['', f"Final test: mean abs. count error {fin['test']['mean_abs_count_error']}, count correlation {fin['test']['count_correlation']}"]
if 'test_tiles_map' in det:
    lines += [f"Detector tile-level test mAP50 {det['test_tiles_map']['mAP50']:.3f}"]
os.makedirs(f'{WORK}/results', exist_ok=True)
Path(f'{WORK}/results/RESULTS.md').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
for f in [RUN / 'results.png'] + sorted(glob.glob(f'{WORK}/results/two_stage/overviews/*.jpg'))[:1]:
    if Path(f).exists():
        display(Show(filename=str(f), width=850))

In [ ]:
# Keep the output small
for p in glob.glob(f'{WORK}/runs/**/*.cache', recursive=True):
    os.remove(p)
!du -sh {WORK}/runs {WORK}/results